In [1]:
PATH_WORK_DIR = ".."

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.chdir(PATH_WORK_DIR)
print(f"DIRECTORY: {os.getcwd()}")

DIRECTORY: c:\Users\jayar\Desktop\바탕 화면\REPO\PROJECT\M2-PJT_TXT


In [4]:
import sys
sys.path.append("src")

# packages

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis

# data

In [ ]:
PATH = "./result/topic/doc_topic_mat_monthly.pkl"
with open(file=PATH, mode="rb") as f:
    df = pickle.load(f)

# functions

In [ ]:
def clr_transform(
    df: pd.DataFrame, 
    eps: float=1e-10,
) -> np.ndarray:
    X = df.values
    X = X + eps
    gmean = np.exp(np.mean(np.log(X), axis=1, keepdims=True))
    X_clr = np.log(X / gmean)
    return X_clr

In [ ]:
def engine(
    df: pd.DataFrame,
    num_factors: int, 
    seed: int,
) -> tuple[list, dict]:
    # FACTOR ANALYSIS ==========
    kwargs = dict(
        n_components=num_factors,
        random_state=seed,
    )
    fa = FactorAnalysis(**kwargs)

    # FACTOR SCORE ==========
    scores = pd.DataFrame(
        data=fa.fit_transform(df),
        index=df.index,
        columns=[f"f{i+1}" for i in range(num_factors)],
    )

    # FACTOR LOADING ==========
    loadings = pd.DataFrame(
        data=fa.components_.T,
        index=df.columns,
        columns=[f"f{i+1}" for i in range(num_factors)],
    )
    loadings.index.name = "topic"

    # LOG LIKELIHOOD ==========
    ll = fa.score(df)

    result = {
        "scores": scores,
        "loadings": loadings,
    }

    return ll, result

# modeling

In [12]:
NUM_FACTORS = range(1,9)
SEED = 42

In [ ]:
kwargs = dict(
    data=clr_transform(df),
    columns=df.columns,
    index=df.index,
)

df = pd.DataFrame(**kwargs)

In [ ]:
scaler = StandardScaler()

kwargs = dict(
    data=scaler.fit_transform(df),
    columns=df.columns,
    index=df.index,
)

df = pd.DataFrame(**kwargs)

In [ ]:
lls = dict()

for k in NUM_FACTORS:
    kwargs = dict(
        df=df,
        num_factors=k, 
        seed=SEED,
    )
    ll, result = engine(**kwargs)

    lls[k] = ll

    PATH = f"./result/factor/factor/f{k}.pkl"
    with open(file=PATH, mode="wb") as f:
        pickle.dump(obj=result, file=f)

    print(
        f"NUM FACTOR: {k}",
        f"LIKELIHOOD: {ll:.4f}",
        sep="\t",
    )

print("FACTOR ANALYSIS FINISHED")

NUM FACTOR: 1	LIKELIHOOD: -20.2594
NUM FACTOR: 2	LIKELIHOOD: -19.8041
NUM FACTOR: 3	LIKELIHOOD: -19.5001
NUM FACTOR: 4	LIKELIHOOD: -19.3968
NUM FACTOR: 5	LIKELIHOOD: -19.2916
NUM FACTOR: 6	LIKELIHOOD: -19.2165
NUM FACTOR: 7	LIKELIHOOD: -19.1432
NUM FACTOR: 8	LIKELIHOOD: -19.0623
FACTOR ANALYSIS FINISHED


# save

In [ ]:
PATH = "./result/factor/lls.pkl"
with open(file=PATH, mode="wb") as f:
    pickle.dump(obj=lls, file=f)